# Settle Up Problem (Dlužníček)

## Motivation

You went on a trip with a group of your friends. All of you shared some expenses, and now it is the time to settle all the debts. It is clear that everyone should pay the same amount; however, people are lazy, and so you want to find the solution which minimizes the number of transactions.

## Input

You are given the following:

* A set of people $P$
* For every person $i \in P$ the cost $c_i$ (i.e., amount of money that $i$ payed)

For the experiments, you may use the following instance:

In [ ]:
!pip install gurobipy
P = set(["A", "B", "C", "D"])
c = {"A": 0, "B": 590, "C": 110, "D": 300}  # c_i is accessed by calling c[i]
sv = sum(c.values())/len(c)  # the settlement value

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 25.4 MB/s eta 0:00:00


Number $sv$ (the settlement value) is the fair price that every person should pay.

## Output

You should find a list of tuples $(x, y, z)$ representing the transactions: person $x$ should pay person $y$ amount of $z$ euros. The number of transactions (i.e., the length of the list) should be minimized.

## Exercise

Implement the ILP model of the problem. You can assume that the settlement value is int (or was rounded).

In [ ]:
import gurobipy as g  # import Gurobi module


# model --------------------------------------------------
m = g.Model()

# - ADD VARIABLES
f = {}
y = {}
for p in P:
    for q in P:
        # f_pq represents how much money should p pay to q
        f[p, q] = m.addVar(lb=0, vtype=g.GRB.CONTINUOUS)
        # y_pq represents whether p should pay to q or not
        y[p, q] = m.addVar(vtype=g.GRB.BINARY)

# - ADD CONSTRAINTS
sv = sum(c.values()) / len(c)
for p in P:
    # what person gets, payed and pays sums up to settlement value
    m.addConstr(g.quicksum(f[p, q] for q in P) + c[p] - g.quicksum(f[q, p] for q in P) == sv)

M = sum(c.values())

for p in P:
    for q in P:
        # link variables y with f
        m.addConstr(f[p, q] <= y[p, q] * M)
        m.addConstr(y[p, q] <= f[p, q])

# - SET OBJECTIVE
# minimize the number of transactions
m.setObjective(g.quicksum(y[p, q] for p in P for q in P), sense=g.GRB.MINIMIZE)

# call the solver -------------------------------------------
m.optimize()

# print the solution -----------------------------------------
print('\nSOLUTION:')
for p in P:
    for q in P:
        if f[p, q].x > 0:
            print("{0} -> {1}: {2}".format(p, q, f[p, q].x))

#print([y[p, q].x for p in P for q in P])
# always round results before casting to int!
#print([int(round(f[p, q].x)) for p in P for q in P])

Restricted license - for non-production use only - expires 2026-11-23
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (linux64 - "Ubuntu 22.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 36 rows, 32 columns and 88 nonzeros
Model fingerprint: 0xb994427b
Variable types: 16 continuous, 16 integer (16 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+03]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e+01, 3e+02]
Found heuristic solution: objective 3.0000000
Presolve removed 8 rows and 8 columns
Presolve time: 0.00s
Presolved: 28 rows, 24 columns, 72 nonzeros
Variable types: 12 continuous, 12 integer (12 binary)

Root relaxation: objective 3.900000e-01, 9 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth In

In [ ]:
# Alternativni model ktery minimalizuje presunute penize (lze ukazat, lze take probrat jak optimalizovat pocet plateb a jako sekundarni kriterium sumu prevedenych penez)

import gurobipy as g  # import Gurobi module
from itertools import permutations

min_value = 10000
big_m = 100000  # max(sum(c.values()), min_value * len([permutations(P, 2)])) - wrong

# model --------------------------------------------------
m = g.Model()

# - ADD VARIABLES
transactions = m.addVars(permutations(P, 2), vtype="I")
print(transactions)

# - ADD CONSTRAINTS
# Add incoming to paid (inc = payto, paywho) - outgoing
m.addConstrs(sum(transactions[p, q] for q in P if p != q) + c[p] - g.quicksum(transactions[q, p] for q in P if p != q)
             == sv for p in P)

# - SET OBJECTIVE
m.setObjective(transactions.sum(), sense=g.GRB.MINIMIZE)

# call the solver -------------------------------------------
m.optimize()

# print the solution -----------------------------------------
print('\nSOLUTION:')
print([str(comb) + ": " + str(transactions[comb].X) for comb in transactions])

{('B', 'A'): <gurobi.Var *Awaiting Model Update*>, ('B', 'D'): <gurobi.Var *Awaiting Model Update*>, ('B', 'C'): <gurobi.Var *Awaiting Model Update*>, ('A', 'B'): <gurobi.Var *Awaiting Model Update*>, ('A', 'D'): <gurobi.Var *Awaiting Model Update*>, ('A', 'C'): <gurobi.Var *Awaiting Model Update*>, ('D', 'B'): <gurobi.Var *Awaiting Model Update*>, ('D', 'A'): <gurobi.Var *Awaiting Model Update*>, ('D', 'C'): <gurobi.Var *Awaiting Model Update*>, ('C', 'B'): <gurobi.Var *Awaiting Model Update*>, ('C', 'A'): <gurobi.Var *Awaiting Model Update*>, ('C', 'D'): <gurobi.Var *Awaiting Model Update*>}
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (linux64 - "Ubuntu 22.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 4 rows, 12 columns and 24 nonzeros
Model fingerprint: 0xa455f8a6
Variable types: 0 continuous, 12 integer (0 binary)
Coefficient statistics:
 